# ST 554 Homework 9
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

In [1]:
#importing required modules
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import when
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

import os
# expanding memory allocation due to previous errors/timeouts
os.environ['SPARK_DRIVER_MEMORY'] = '10g'
os.environ['SPARK_EXECUTOR_MEMORY'] = '10g'

## Reading in Data
For this assignment, I chose to use a dataset on Florida real estate properties sold in 2026. I downloaded this dataset from Kaggle, and it is linked [here](https://www.kaggle.com/datasets/kanchana1990/florida-real-estate-sold-dataset-2026). The data file will also be available in my [Github](https://github.com/jlloring/ST-554_JLoring), titled `florida_real_estate_sold_properties_utlimate.csv`.

### Creating a Spark Session
The code below creates a spark session for use with reading in our data and building our models.

In [2]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 14:37:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


The code below only allows error messages to print out going forward.

In [3]:
spark.sparkContext.setLogLevel("ERROR")

### Importing Data
The code below reads in our data using `pandas`. The `.dropna()` method is also used to drop rows where `NaN` values are present within certain variables. I made sure that the resulting number of rows dropped did not result in significant data loss. Finally, the `.fillna()` method is used with context clues to fill missing values with numeric values for certain variables. For the `stories` variable, `NaN` will be changed to 1.0. For the `garage` variable, `NaN` will be changed to 0.

In [4]:
house_data = pd.read_csv("florida_real_estate_sold_properties_ultimate.csv") \
             .dropna(subset=['sqft', 'year_built', 'beds', 'baths_full', 'listPrice'])

house_data['stories'] = house_data['stories'].fillna(1.0)
house_data['garage'] = house_data['garage'].fillna(0.0)

house_data.head()

,type,sub_type,listPrice,lastSoldPrice,sqft,stories,beds,baths,baths_full,baths_full_calc,garage,year_built,zip,sanitized_text
0,single_family,NaN,630000.0,605000,2274.0,1.0,2.0,3.0,2.0,2.0,2.0,2007.0,33446.0,"Beautiful 2 Bedroom + Den, 2.5 Bath Home - Mov..."
1,single_family,NaN,289000.0,285000,2170.0,1.0,3.0,2.0,2.0,2.0,2.0,1980.0,33876.0,Welcome to Florida living at its best! This 3-...
2,condos,condo,449000.0,425000,1722.0,1.0,3.0,2.0,2.0,2.0,2.0,2016.0,33913.0,Best Value in Casella and priced to sell... St...
3,single_family,NaN,599000.0,596000,1699.0,1.0,3.0,3.0,3.0,3.0,0.0,1952.0,33009.0,"Beautifully renovated 3-bedroom, 3-bathroom ho..."
4,single_family,NaN,173500.0,165000,640.0,1.0,1.0,1.0,1.0,1.0,0.0,1971.0,32118.0,Experience the ultimate beachfront lifestyle i...


Now, we convert our `pandas dataframe` to a `spark` SQL style dataframe. We also create a variable, `sgl_fmily_ind`, that will be used later. This variable takes a value of 1 for single family homes and 0 otherwise.

In [5]:
FL_houses = spark.createDataFrame(house_data)
FL_houses = FL_houses.withColumn("sgl_fmily_ind", 
                                 when(FL_houses["type"] == "single_family", 1) \
                                 .otherwise(0))
FL_houses.show(5)

+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+-------------+
|         type|sub_type|listPrice|lastSoldPrice|  sqft|stories|beds|baths|baths_full|baths_full_calc|garage|year_built|    zip|      sanitized_text|sgl_fmily_ind|
+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+-------------+
|single_family|     NaN| 630000.0|       605000|2274.0|    1.0| 2.0|  3.0|       2.0|            2.0|   2.0|    2007.0|33446.0|Beautiful 2 Bedro...|            1|
|single_family|     NaN| 289000.0|       285000|2170.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    1980.0|33876.0|Welcome to Florid...|            1|
|       condos|   condo| 449000.0|       425000|1722.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    2016.0|33913.0|Best Value in Cas...|            0|
|single_family|     Na

## Splitting the Data, Metrics, and Models
This section involves:
- splitting the data into a training and test set using spark MLlib
- choosing and describing a metric for judging the fitted models
- fitting three different classes of models and describing them

### Splitting the Data
The code below uses spark MLlib to split the data into a training and test set. This is done by using the `.randomSplit()` method on a `spark` SQL style dataframe.

In [6]:
train, test = FL_houses.randomSplit([0.8,0.2], seed = 44)
print('Number of training observations:', train.count())
print('Number of test observations:', test.count())

Number of training observations: 8010
Number of test observations: 2061


### Choosing a Metric
For this assignment, I will choose the root mean square error (RMSE) for judging the models. RMSE is a very common metric and is easy to interpret. The lower the RMSE, the better the model fit. The mean square error (MSE) is the average of the difference between observed values and values predicted by the model. We want to minimze MSE because we want our model predictions to be close to what is observed! As such, the RMSE is simply the square-root of the MSE.

### Classes of Models
Since I am interested in using `lastSoldPrice` as the response (more specifically, the log of `lastSoldPrice`), I will fit three different classes of *regression* models. The following models will be fit:
- A **linear regression model**
- A **decision tree regression model**
- A **random forest regression model**

## Model Fitting
In this section, we use Spark MLlib to fit the three different classes of models described previously to the training data. A pipeline in `pyspark` will be set up for each model. Cross validation will be used to choose the best model for each model type. These best models from each class will be compared using RMSE on the test data, and those results will be used to select the best model overall!

### Model 1: Linear Regression Model
The first model we will fit is a linear regression model that uses the following features to predict log(`lastSoldPrice`):
- `sqft`
- a multi-story indicator variable that will be calculated using the `stories` variable
    - will be called `multi_story_ind`
- `year_built`
- `beds`
- `baths_full`

**Note:** This is the model that will feature 4 transformations in the pipeline!

We first need to create our multi-story indicator variable. This can be done using `Binarizer()`. We will use a threshold of 1.5 since anything lower would clearly indicate that the home is not multi-story, whereas anything larger would clearly indicate that the home is multi-story.

In [7]:
binaryTrans1 = Binarizer(threshold = 1.5, inputCol = "stories", outputCol = "multi_story_ind")
binaryTrans1.transform(FL_houses).show(5)

+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+-------------+---------------+
|         type|sub_type|listPrice|lastSoldPrice|  sqft|stories|beds|baths|baths_full|baths_full_calc|garage|year_built|    zip|      sanitized_text|sgl_fmily_ind|multi_story_ind|
+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+-------------+---------------+
|single_family|     NaN| 630000.0|       605000|2274.0|    1.0| 2.0|  3.0|       2.0|            2.0|   2.0|    2007.0|33446.0|Beautiful 2 Bedro...|            1|            0.0|
|single_family|     NaN| 289000.0|       285000|2170.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    1980.0|33876.0|Welcome to Florid...|            1|            0.0|
|       condos|   condo| 449000.0|       425000|1722.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2

Next, we use the `SQLTransformer()` to select the columns that we want and rename the response, the log of `lastSoldPrice`, as `label`.

In [8]:
sqlTrans1 = SQLTransformer(
    statement = """
                SELECT sqft, multi_story_ind, year_built, beds, baths_full, log(lastSoldPrice) as label FROM __THIS__
                """
            )

sqlTrans1.transform(
    binaryTrans1.transform(FL_houses)
).show(5)

+------+---------------+----------+----+----------+------------------+
|  sqft|multi_story_ind|year_built|beds|baths_full|             label|
+------+---------------+----------+----+----------+------------------+
|2274.0|            0.0|    2007.0| 2.0|       2.0|13.312983737012978|
|2170.0|            0.0|    1980.0| 3.0|       2.0|12.560244459250788|
|1722.0|            0.0|    2016.0| 3.0|       2.0|12.959844447906553|
|1699.0|            0.0|    1952.0| 3.0|       3.0|13.297995946047486|
| 640.0|            0.0|    1971.0| 1.0|       1.0|12.013700752882718|
+------+---------------+----------+----+----------+------------------+
only showing top 5 rows


Continuing on, we need to put our model features into a single column called `features`. We can do this with `VectorAssembler()`.

In [9]:
assembler1 = VectorAssembler(
                inputCols = ["sqft", "multi_story_ind", "year_built", "beds", "baths_full"],
                outputCol = "features",
                handleInvalid = "keep"
             )

assembler1.transform(
    sqlTrans1.transform(
        binaryTrans1.transform(FL_houses)
    )
).show(5)

+------+---------------+----------+----+----------+------------------+--------------------+
|  sqft|multi_story_ind|year_built|beds|baths_full|             label|            features|
+------+---------------+----------+----+----------+------------------+--------------------+
|2274.0|            0.0|    2007.0| 2.0|       2.0|13.312983737012978|[2274.0,0.0,2007....|
|2170.0|            0.0|    1980.0| 3.0|       2.0|12.560244459250788|[2170.0,0.0,1980....|
|1722.0|            0.0|    2016.0| 3.0|       2.0|12.959844447906553|[1722.0,0.0,2016....|
|1699.0|            0.0|    1952.0| 3.0|       3.0|13.297995946047486|[1699.0,0.0,1952....|
| 640.0|            0.0|    1971.0| 1.0|       1.0|12.013700752882718|[640.0,0.0,1971.0...|
+------+---------------+----------+----+----------+------------------+--------------------+
only showing top 5 rows


We are now ready to set up a pipeline for this model and fit a multiple linear regression (MLR) model. A few important notes:
- The `Pipeline()` function from `pyspark.ml` will be used to set up our sequence of transformations and estimators
- Since `LinearRegression()` does regularized regression, we will provide a list of values for both `regParam` and `elasticNetParam` so that cross-validation can be used to select the best model!

In [10]:
# set up instance of linear regression
lr = LinearRegression()

# build parameter grid
paramGrid1 = ParamGridBuilder() \
             .addGrid(lr.regParam, [0, 0.25, 0.5, 0.75, 1]) \
             .addGrid(lr.elasticNetParam, [0, 0.5, 0.75, 0.9, 1]) \
             .build()

# create pipeline
pipeline1 = Pipeline(stages = [binaryTrans1, sqlTrans1, assembler1, lr])

# set up cross validation with pipeline
crossval1 = CrossValidator(estimator = pipeline1,
                           estimatorParamMaps = paramGrid1,
                           evaluator = RegressionEvaluator(metricName = 'rmse'),
                           numFolds = 5,
                           seed = 44)

# fit the model using cross-validation to choose the best model
cvModel1 = crossval1.fit(train)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `regParam` and `elasticNetParam`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters).

In [11]:
my_list1 = []

for i in range(len(paramGrid1)):
    my_list1.append([cvModel1.avgMetrics[i], paramGrid1[i].values()])
    
my_list1.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list1

[[0.6303350852930365, dict_values([0.0, 0.9])],
 [0.630335085293038, dict_values([0.0, 0.5])],
 [0.630335085293038, dict_values([0.0, 0.75])],
 [0.6303350852930392, dict_values([0.0, 1.0])],
 [0.6303350852930407, dict_values([0.0, 0.0])],
 [0.6347609174216456, dict_values([0.25, 0.0])],
 [0.6445414360362022, dict_values([0.5, 0.0])],
 [0.6489772584706973, dict_values([0.25, 0.5])],
 [0.6539444424732104, dict_values([0.75, 0.0])],
 [0.6626790053819067, dict_values([1.0, 0.0])],
 [0.6628897037287169, dict_values([0.25, 0.75])],
 [0.6726763413239052, dict_values([0.25, 0.9])],
 [0.6796431166510218, dict_values([0.25, 1.0])],
 [0.6980802124575147, dict_values([0.5, 0.5])],
 [0.7456526331476178, dict_values([0.5, 0.75])],
 [0.758462534509481, dict_values([0.75, 0.5])],
 [0.7787562790162627, dict_values([0.5, 0.9])],
 [0.8062720449624585, dict_values([0.5, 1.0])],
 [0.8119031285393437, dict_values([1.0, 0.5])],
 [0.8213000404434648, dict_values([0.75, 0.75])],
 [0.8213000404434648, dict_valu

Thus, the best multiple linear regression model is one with a regularization parameter of 0 and an elastic net parameter of 1.0. This tells us that we are fitting an ordinary multiple linear regression model since we have no regularization, regardless of what our elastic net parameter is! We can now print the intercept and coefficients of this best model.

In [12]:
# need to use indexing since the model is the last stage of the pipeline
print('Intercept:', cvModel1.bestModel.stages[-1].intercept)
print('sqft, multi_story_ind, year_built, beds, baths_full coeffs:', cvModel1.bestModel.stages[-1].coefficients)
print('Best RMSE:', round(my_list1[0][0], 5))

Intercept: 12.784623434190765
sqft, multi_story_ind, year_built, beds, baths_full coeffs: [0.0004655190466502769,-0.048656686462162375,-0.000578241982138113,-0.013407181325996031,0.20260005210723253]
Best RMSE: 0.63034


### Model 2: Decision Tree Regression Model
The second model we will fit is a decision tree regression model that uses the following features to predict log(`lastSoldPrice`):
- `stories`
- `beds`
- `baths_full`
- `garage`

We start by using the `SQLTransformer()` to select the columns that we want and rename the response, the log of `lastSoldPrice`, as `label`.

In [13]:
sqlTrans2 = SQLTransformer(
    statement = """
                SELECT stories, beds, baths_full, garage, log(lastSoldPrice) as label FROM __THIS__
                """
            )

sqlTrans2.transform(FL_houses).show(5)

+-------+----+----------+------+------------------+
|stories|beds|baths_full|garage|             label|
+-------+----+----------+------+------------------+
|    1.0| 2.0|       2.0|   2.0|13.312983737012978|
|    1.0| 3.0|       2.0|   2.0|12.560244459250788|
|    1.0| 3.0|       2.0|   2.0|12.959844447906553|
|    1.0| 3.0|       3.0|   0.0|13.297995946047486|
|    1.0| 1.0|       1.0|   0.0|12.013700752882718|
+-------+----+----------+------+------------------+
only showing top 5 rows


Next, we need to put our model features into a single column called `features`. We can do this with `VectorAssembler()`.

In [14]:
assembler2 = VectorAssembler(
                inputCols = ["stories", "beds", "baths_full", "garage"],
                outputCol = "features",
                handleInvalid = "keep"
             )

assembler2.transform(
    sqlTrans2.transform(FL_houses)
).show(5)

+-------+----+----------+------+------------------+-----------------+
|stories|beds|baths_full|garage|             label|         features|
+-------+----+----------+------+------------------+-----------------+
|    1.0| 2.0|       2.0|   2.0|13.312983737012978|[1.0,2.0,2.0,2.0]|
|    1.0| 3.0|       2.0|   2.0|12.560244459250788|[1.0,3.0,2.0,2.0]|
|    1.0| 3.0|       2.0|   2.0|12.959844447906553|[1.0,3.0,2.0,2.0]|
|    1.0| 3.0|       3.0|   0.0|13.297995946047486|[1.0,3.0,3.0,0.0]|
|    1.0| 1.0|       1.0|   0.0|12.013700752882718|[1.0,1.0,1.0,0.0]|
+-------+----+----------+------+------------------+-----------------+
only showing top 5 rows


We are now ready to set up a pipeline for this model and fit a decision tree regression model. A few important notes:
- The `Pipeline()` function from `pyspark.ml` will be used to set up our sequence of transformations and estimators
- A list of values will be provided for the `maxDepth` and `minInstancesPerNode` options so that cross-validation can be used to select the best model!

In [15]:
# set up instance of decision tree
dt = DecisionTreeRegressor(seed = 44)

# build parameter grid
paramGrid2 = ParamGridBuilder() \
             .addGrid(dt.maxDepth, [2, 3, 4, 5, 6, 7, 8]) \
             .addGrid(dt.minInstancesPerNode, [2, 3, 5, 10]) \
             .build()

# create pipeline
pipeline2 = Pipeline(stages = [sqlTrans2, assembler2, dt])

# set up cross validation with pipeline
crossval2 = CrossValidator(estimator = pipeline2,
                           estimatorParamMaps = paramGrid2,
                           evaluator = RegressionEvaluator(metricName = 'rmse'),
                           numFolds = 5,
                           seed = 44)

# fit the model using cross-validation to choose the best model
cvModel2 = crossval2.fit(train)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `maxDepth` and `minInstancesPerNode`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters).

In [16]:
my_list2 = []

for i in range(len(paramGrid2)):
    my_list2.append([cvModel2.avgMetrics[i], paramGrid2[i].values()])
    
my_list2.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list2

[[0.6279351085660048, dict_values([7, 10])],
 [0.6281803459618187, dict_values([8, 10])],
 [0.6287213533427798, dict_values([5, 5])],
 [0.6287991054554432, dict_values([5, 10])],
 [0.6288407587990921, dict_values([6, 5])],
 [0.6288496219430109, dict_values([6, 10])],
 [0.6292002104898842, dict_values([8, 5])],
 [0.6292759100236609, dict_values([6, 3])],
 [0.6294202234804137, dict_values([7, 5])],
 [0.6298256190656268, dict_values([5, 3])],
 [0.6301929010318843, dict_values([5, 2])],
 [0.6310870593938807, dict_values([7, 3])],
 [0.6314345187590253, dict_values([6, 2])],
 [0.6315204130510718, dict_values([8, 3])],
 [0.632530244733196, dict_values([4, 5])],
 [0.6325733909622554, dict_values([4, 3])],
 [0.6326182255512464, dict_values([4, 10])],
 [0.6326565369216047, dict_values([4, 2])],
 [0.6331302354021803, dict_values([7, 2])],
 [0.6338194460446404, dict_values([8, 2])],
 [0.6411727352781152, dict_values([3, 2])],
 [0.6411727352781152, dict_values([3, 5])],
 [0.6411727352781152, dict_v

Thus, the best decision tree regression model is one with a max depth of 7 and and 10 minimum instances per node. We will now print the total number of nodes in the decision tree.

In [17]:
# need to use indexing since the model is the last stage of the pipeline
print('Total Number of Nodes:', cvModel2.bestModel.stages[-1].numNodes)
print('Best RMSE:', round(my_list2[0][0], 5))

Total Number of Nodes: 129
Best RMSE: 0.62794


### Model 3: Random Forest Regression Model
The third and final model we will fit is a random forest regression model that uses the following features to predict log(`lastSoldPrice`):
- `sgl_fmily_ind`, the single-family home indicator variable created towards the beginning of the notebook
- the log of the `listPrice`
    - will be called `log_list_price`
- `sqft`
- `beds`
- `baths`
    - this is the total number of all baths, including half-baths

We can now use the `SQLTransformer()` to select the columns that we want and rename the response, the log of `lastSoldPrice`, as `label`.

In [18]:
sqlTrans3 = SQLTransformer(
    statement = """
                SELECT sgl_fmily_ind, log(listPrice) as log_list_price, sqft,
                       beds, baths, log(lastSoldPrice) as label FROM __THIS__
                """
            )

sqlTrans3.transform(FL_houses).show(5)

+-------------+------------------+------+----+-----+------------------+
|sgl_fmily_ind|    log_list_price|  sqft|beds|baths|             label|
+-------------+------------------+------+----+-----+------------------+
|            1|13.353475098367715|2274.0| 2.0|  3.0|13.312983737012978|
|            1| 12.57418196709457|2170.0| 3.0|  2.0|12.560244459250788|
|            0| 13.01477816672439|1722.0| 3.0|  2.0|12.959844447906553|
|            1|13.303016877097587|1699.0| 3.0|  3.0|13.297995946047486|
|            1|12.063932878369052| 640.0| 1.0|  1.0|12.013700752882718|
+-------------+------------------+------+----+-----+------------------+
only showing top 5 rows


Next, we need to put our model features into a single column called `features`. We can do this with `VectorAssembler()`.

In [19]:
assembler3 = VectorAssembler(
                inputCols = ["sgl_fmily_ind", "log_list_price", "sqft", "beds", "baths"],
                outputCol = "features",
                handleInvalid = "keep"
             )

assembler3.transform(
    sqlTrans3.transform(FL_houses)
).show(5)

+-------------+------------------+------+----+-----+------------------+--------------------+
|sgl_fmily_ind|    log_list_price|  sqft|beds|baths|             label|            features|
+-------------+------------------+------+----+-----+------------------+--------------------+
|            1|13.353475098367715|2274.0| 2.0|  3.0|13.312983737012978|[1.0,13.353475098...|
|            1| 12.57418196709457|2170.0| 3.0|  2.0|12.560244459250788|[1.0,12.574181967...|
|            0| 13.01477816672439|1722.0| 3.0|  2.0|12.959844447906553|[0.0,13.014778166...|
|            1|13.303016877097587|1699.0| 3.0|  3.0|13.297995946047486|[1.0,13.303016877...|
|            1|12.063932878369052| 640.0| 1.0|  1.0|12.013700752882718|[1.0,12.063932878...|
+-------------+------------------+------+----+-----+------------------+--------------------+
only showing top 5 rows


We are now ready to set up a pipeline for this model and fit a random forest regression model. A few important notes:
- The `Pipeline()` function from `pyspark.ml` will be used to set up our sequence of transformations and estimators
- A list of values will be provided for the `maxDepth` and `minInstancesPerNode` options so that cross-validation can be used to select the best model!

**Note:** For this particular model, I used a small range of parameter choices due to memory issues.

In [20]:
# set up instance of random forest model
rf = RandomForestRegressor(seed = 44)

# build parameter grid
paramGrid3 = ParamGridBuilder() \
             .addGrid(rf.maxDepth, [10, 15, 20]) \
             .addGrid(rf.minInstancesPerNode, [2, 3, 5, 10]) \
             .build()

# create pipeline
pipeline3 = Pipeline(stages = [sqlTrans3, assembler3, rf])

# set up cross validation with pipeline
crossval3 = CrossValidator(estimator = pipeline3,
                           estimatorParamMaps = paramGrid3,
                           evaluator = RegressionEvaluator(metricName = 'rmse'),
                           numFolds = 5,
                           seed = 44)

# fit the model using cross-validation to choose the best model
cvModel3 = crossval3.fit(train)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `maxDepth` and `minInstancesPerNode`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters).

In [21]:
my_list3 = []

for i in range(len(paramGrid3)):
    my_list3.append([cvModel3.avgMetrics[i], paramGrid3[i].values()])
    
my_list3.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list3

[[0.2528279038933645, dict_values([15, 5])],
 [0.25282871615280683, dict_values([20, 5])],
 [0.2533889424909304, dict_values([10, 5])],
 [0.2548530930238687, dict_values([20, 2])],
 [0.254857124262141, dict_values([15, 2])],
 [0.2551163932916662, dict_values([10, 2])],
 [0.25524307983711647, dict_values([20, 3])],
 [0.2552524536701656, dict_values([15, 3])],
 [0.2560232809037094, dict_values([10, 3])],
 [0.2565382578455607, dict_values([20, 10])],
 [0.25653826862033574, dict_values([15, 10])],
 [0.25695294194838236, dict_values([10, 10])]]

Thus, the best random forest regression model is one with a max depth of 15 and and 5 minimum instances per node. We will now print the total number of nodes in the random forest.

In [22]:
# need to use indexing since the model is the last stage of the pipeline
print('Total Number of Trees:', cvModel3.bestModel.stages[-1]._java_obj.getNumTrees())
print('Best RMSE:', round(my_list3[0][0], 5))

Total Number of Trees: 20
Best RMSE: 0.25283


### **Model Fitting Recap**
We have now fit three different classes of models:
1. a multiple linear regression model
2. a decision tree regression model
3. a random forest regression model

As a reminder, the RMSE for the best model selected from each class is as follows:

In [23]:
print('Training RMSE for Best Multiple Linear Regression Model:', round(my_list1[0][0], 5))
print('Training RMSE for Best Decision Tree Regression Model:', round(my_list2[0][0], 5))
print('Training RMSE for Best Random Forest Regression Model:', round(my_list3[0][0], 5))

Best Training RMSE for Multiple Linear Regression Model: 0.63034
Best Training RMSE for Decision Tree Regression Model: 0.62794
Best Training RMSE for Random Forest Regression Model: 0.25283


Now, let's see how these models do on the test data!

## Model Testing
In this section, we use Spark MLlib to evaluate the best models from each class on the test set and state which overall model
is deemed the best. We will start by computing the test RMSEs for the best multiple linear regression , the best decision tree regression model, and the best random forest regression model.

In [24]:
test_error1 = RegressionEvaluator().evaluate(cvModel1.transform(test))
test_error2 = RegressionEvaluator().evaluate(cvModel2.transform(test))
test_error3 = RegressionEvaluator().evaluate(cvModel3.transform(test))

print('Test RMSE for Best Multiple Linear Regression Model:', round(test_error1, 5))
print('Test RMSE for Best Decision Tree Regression Model:', round(test_error2, 5))
print('Test RMSE for Best Random Forest Regression Model:', round(test_error3, 5))

Best Test RMSE for Multiple Linear Regression Model: 0.62466
Best Test RMSE for Decision Tree Regression Model: 0.63253
Best Test RMSE for Random Forest Regression Model: 0.29058


Therefore, we see that the best overall model is the _**random forest regression model**_ with the following parameters:
- Max Depth = 15
- Minimum Instances Per Node = 5
- Resulting RMSE on the Test Set: 0.29058